# 03 · Pathway enrichment

**Question:** Which biological pathways are over-represented among macrocephaly genes? In particular, do **PI3K/AKT/mTOR** and **Ras/MAPK** signalling come out of the data, as the literature suggests?

**Tool:** [g:Profiler](https://biit.cs.ut.ee/gprofiler/) through its Python package. It tests GO biological processes, KEGG and Reactome in one call.

**Background choice:** the gene universe is all HPO-annotated genes, not the whole genome. Our genes came from HPO, so comparing them against the whole genome would make "disease genes in general" look enriched. This way, we find what is specific to macrocephaly genes among disease genes.

**Input:** `results/gene_sets/` from notebook 01  
**Outputs:** `results/enrichment_macrocephaly.csv`, `results/enrichment_shared37.csv`, `figures/enrichment_top_terms.png`

## 1. Setup
Upload `results_02.zip` (or `results_01.zip`) through the Files panel first, then run.

In [ ]:
!pip install -q gprofiler-official
!unzip -q -o results_02.zip 2>/dev/null || unzip -q -o results_01.zip

import os
from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gprofiler import GProfiler

os.makedirs("figures", exist_ok=True)

def read_set(path):
    with open(path) as f:
        return sorted({line.strip() for line in f if line.strip()})

macro = read_set("results/gene_sets/macrocephaly.txt")
background = read_set("results/gene_sets/background_genes.txt")
print(f"Macrocephaly genes: {len(macro)} | Background: {len(background):,}")

## 2. Run the enrichment
- `significance_threshold_method="g_SCS"` is g:Profiler's own multiple-testing correction, designed for the nested structure of GO terms.
- `no_evidences=False` returns which of our genes fall in each term, so we can see the genes behind each pathway.

In [ ]:
gp = GProfiler(return_dataframe=True)

def run_enrichment(genes):
    res = gp.profile(
        organism="hsapiens",
        query=genes,
        sources=["GO:BP", "KEGG", "REAC"],
        background=background,
        domain_scope="custom",
        significance_threshold_method="g_SCS",
        user_threshold=0.05,
        no_evidences=False,
    )
    keep = ["source", "native", "name", "p_value", "term_size",
            "intersection_size", "precision", "recall", "intersections"]
    res = res[keep].sort_values("p_value").reset_index(drop=True)
    res["intersections"] = res["intersections"].apply(lambda g: ", ".join(sorted(g)))
    return res

enr = run_enrichment(macro)
RUN_DATE = date.today().isoformat()
enr.to_csv("results/enrichment_macrocephaly.csv", index=False)

print(f"Significant terms: {len(enr)} (run on {RUN_DATE})")
print(enr["source"].value_counts())
enr.head(15)[["source", "name", "p_value", "intersection_size", "term_size"]]

## 3. Do the expected pathways appear?
This searches the significant terms for the signalling pathways named in the original UG thesis.

In [ ]:
PATTERNS = {
    "PI3K / AKT / mTOR": r"PI3K|phosphatidylinositol 3-kinase|\bAKT\b|mTOR|TOR signaling",
    "Ras / MAPK":        r"\bRas\b|RAS|RAF|MAPK|ERK",
    "Wnt":               r"Wnt|WNT|destruction complex|AXIN|beta-catenin",
    "Hedgehog":          r"[Hh]edgehog",
    "Chromatin":         r"chromatin|histone",
}

for label, pat in PATTERNS.items():
    hits = enr[enr["name"].str.contains(pat, regex=True)]
    print(f"\n=== {label}: {len(hits)} significant terms ===")
    if len(hits):
        print(hits.head(5)[["source", "name", "p_value", "intersection_size"]].to_string(index=False))

**How to read this:** a pathway "comes out of the data" if it appears here as significant. If one of them does not appear, that is a real finding too, and worth reporting honestly rather than hiding.

## 4. Figure: top terms from each source

In [ ]:
TOP = 8
plot_df = (enr.groupby("source", group_keys=False)
              .apply(lambda d: d.nsmallest(TOP, "p_value")))
plot_df["neg_log10_p"] = -np.log10(plot_df["p_value"])

sources = [s for s in ["GO:BP", "KEGG", "REAC"] if s in plot_df["source"].unique()]
fig, axes = plt.subplots(len(sources), 1, figsize=(8, 2.2 * len(sources) + 1))
if len(sources) == 1:
    axes = [axes]

for ax, src in zip(axes, sources):
    d = plot_df[plot_df["source"] == src].sort_values("neg_log10_p")
    names = [n if len(n) <= 55 else n[:52] + "..." for n in d["name"]]
    ax.barh(names, d["neg_log10_p"], color="#4C72B0")
    ax.set_title(src, loc="left", fontsize=10, fontweight="bold")
    ax.tick_params(axis="y", labelsize=8)
    for y, (_, r) in enumerate(d.iterrows()):
        ax.text(r["neg_log10_p"] + 0.1, y, str(r["intersection_size"]), va="center", fontsize=7)

axes[-1].set_xlabel("-log10(adjusted p)   (numbers = macrocephaly genes in term)")
fig.suptitle("Pathway enrichment of macrocephaly genes", y=1.0)
plt.tight_layout()
plt.savefig("figures/enrichment_top_terms.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. The 37 genes shared with all three phenotypes
These were found in notebook 02. Running enrichment on this small set asks: what biology connects macrocephaly to microcephaly, autism *and* seizures?  
With only 37 genes, expect fewer significant terms.

In [ ]:
shared = pd.read_csv("results/shared_genes.csv")
shared37 = sorted(shared.loc[shared["n_other_phenotypes"] == 3, "gene"])
print(f"{len(shared37)} genes:", ", ".join(shared37))

enr37 = run_enrichment(shared37)
enr37.to_csv("results/enrichment_shared37.csv", index=False)
print(f"\nSignificant terms: {len(enr37)}")
enr37.head(15)[["source", "name", "p_value", "intersection_size", "intersections"]]

## 6. Note on reproducibility
g:Profiler updates its databases a few times a year, so exact p-values will shift slightly between runs. So the run date and settings are saved in a small text file next to the results.

In [ ]:
with open("results/enrichment_run_info.txt", "w") as f:
    f.write(f"g:Profiler run date: {RUN_DATE}\n")
    f.write("Sources: GO:BP, KEGG, REAC | correction: g_SCS | threshold: 0.05\n")
    f.write(f"Background: {len(background)} HPO-annotated genes (domain_scope=custom)\n")
print(open("results/enrichment_run_info.txt").read())

## 7. Download the results

In [ ]:
!zip -qr results_03.zip results/ figures/
from google.colab import files
files.download("results_03.zip")